# Set-3 Online Food Delivery Analytics: Q1 Complete Solution
**Run all:** `Kernel -> Restart & Run All`. Needs `Food_Delivery_Data.xlsx` (sheets: Customers, Restaurants, Orders) in the same folder.

## 0. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cust = pd.read_excel('Food_Delivery_Data.xlsx', sheet_name='Customers')
rest = pd.read_excel('Food_Delivery_Data.xlsx', sheet_name='Restaurants')
orders = pd.read_excel('Food_Delivery_Data.xlsx', sheet_name='Orders')

print('Shapes (rows, cols):', cust.shape, rest.shape, orders.shape)
for d in (cust, rest, orders):
    display(d.head(3))
    print(d.isna().sum(), '\n')
print('Duplicate rows:', cust.duplicated().sum(), rest.duplicated().sum(), orders.duplicated().sum())

## 1. Missing entries (App_Rating, Preferred_Cuisine, Discount)
- App_Rating is a number -> **median**
- Preferred_Cuisine is text -> **mode** (most common cuisine)
- Discount missing means no discount was given -> **0**

In [ ]:
cust['App_Rating'] = cust['App_Rating'].fillna(cust['App_Rating'].median())
cust['Preferred_Cuisine'] = cust['Preferred_Cuisine'].fillna(cust['Preferred_Cuisine'].mode()[0])
orders['Discount'] = orders['Discount'].fillna(0)

print(cust[['App_Rating', 'Preferred_Cuisine']].isna().sum())
print('Discount missing:', orders['Discount'].isna().sum())

## 2. Duplicate records (Customer_ID, Restaurant_ID, Order_ID)
A missing ID cannot be linked to anything, so those rows are dropped. Then each ID must appear only once in its own table.

In [ ]:
# rows without an ID are unusable
cust = cust.dropna(subset=['Customer_ID'])
rest = rest.dropna(subset=['Restaurant_ID'])
orders = orders.dropna(subset=['Order_ID'])

# one row per ID (keep the first)
cust = cust.drop_duplicates(subset=['Customer_ID'])
rest = rest.drop_duplicates(subset=['Restaurant_ID'])
orders = orders.drop_duplicates(subset=['Order_ID'])

print(cust.shape, rest.shape, orders.shape)
print('Unique?', cust['Customer_ID'].is_unique, rest['Restaurant_ID'].is_unique, orders['Order_ID'].is_unique)

## 3. Order_Date and Order_Time -> proper formats
The file mixes `2025-03-05` and `05-03-2025` for dates, and `14:35` and `02:35 PM` for times. Try each format, then combine the results.

In [ ]:
# --- Order_Date ---
s = orders['Order_Date'].astype(str).str.strip()
d1 = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
d2 = pd.to_datetime(s, format='%d-%m-%Y', errors='coerce')
orders['Order_Date'] = d1.fillna(d2)

# --- Order_Time ---
t = orders['Order_Time'].astype(str).str.strip()
t1 = pd.to_datetime(t, format='%H:%M', errors='coerce')
t2 = pd.to_datetime(t, format='%I:%M %p', errors='coerce')
orders['Order_Time'] = t1.fillna(t2)

print('Dates not converted:', orders['Order_Date'].isna().sum())
print('Times not converted:', orders['Order_Time'].isna().sum())

# new columns for analysis
orders['Order_Month'] = orders['Order_Date'].dt.month_name()
orders['Order_Day'] = orders['Order_Date'].dt.day_name()
orders['Order_Hour'] = orders['Order_Time'].dt.hour
orders['Order_Time'] = orders['Order_Time'].dt.strftime('%H:%M')      # clean text like 14:35

orders['Time_Slot'] = pd.cut(orders['Order_Hour'], bins=[0, 11, 16, 20, 24],
                             labels=['Morning', 'Lunch', 'Evening', 'Night'], right=False)

print(orders[['Order_Date', 'Order_Time', 'Order_Month', 'Order_Day', 'Order_Hour', 'Time_Slot']].head())
print(orders.dtypes[['Order_Date', 'Order_Time']])

## 4. Verify and prepare Final_Amount
Rule: `Final_Amount = Food_Amount + Delivery_Fee - Discount`. First make the three inputs valid.

In [ ]:
# check inputs
print(orders[['Food_Amount', 'Delivery_Fee', 'Discount']].describe())

In [ ]:
# Food_Amount: zero / negative is invalid -> median, then cap huge outliers (IQR rule)
orders.loc[orders['Food_Amount'] <= 0, 'Food_Amount'] = np.nan
orders['Food_Amount'] = orders['Food_Amount'].fillna(orders['Food_Amount'].median())
q1, q3 = orders['Food_Amount'].quantile(0.25), orders['Food_Amount'].quantile(0.75)
orders['Food_Amount'] = orders['Food_Amount'].clip(upper=q3 + 1.5 * (q3 - q1))

# Delivery_Fee cannot be negative
orders.loc[orders['Delivery_Fee'] < 0, 'Delivery_Fee'] = 0

# Discount cannot be negative or more than the food amount
orders.loc[orders['Discount'] < 0, 'Discount'] = 0
orders.loc[orders['Discount'] > orders['Food_Amount'], 'Discount'] = orders['Food_Amount']

# how many Final_Amount values were wrong?
expected = orders['Food_Amount'] + orders['Delivery_Fee'] - orders['Discount']
print('Wrong Final_Amount before fix:', (orders['Final_Amount'] != expected).sum())

# recalculate
orders['Final_Amount'] = expected
print('Wrong Final_Amount after fix :', (orders['Final_Amount'] != expected).sum())

## 4b. Tidy text columns used in Power BI (extra, small)

In [ ]:
for col in ['Payment_Mode', 'Order_Status']:
    orders[col] = orders[col].str.strip().str.title()
orders['Payment_Mode'] = orders['Payment_Mode'].replace({'Upi': 'UPI'})

print(orders['Payment_Mode'].value_counts(), '\n')
print(orders['Order_Status'].value_counts())

## 5. Validate with Matplotlib (Delivery_Time_Min, Delivery_Distance_KM, Food_Amount, Customer_Rating)
First look at the raw problems, then fix impossible values, then plot again.

In [ ]:
# BEFORE fixing: boxplots show the outliers
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].boxplot(orders['Delivery_Time_Min']);      ax[0].set_title('Delivery Time (before)')
ax[1].boxplot(orders['Delivery_Distance_KM']);   ax[1].set_title('Distance KM (before)')
ax[2].boxplot(orders['Customer_Rating']);        ax[2].set_title('Customer Rating (before)')
plt.tight_layout(); plt.show()

In [ ]:
# Fix impossible values -> NaN -> median
orders.loc[(orders['Delivery_Time_Min'] < 5) | (orders['Delivery_Time_Min'] > 180), 'Delivery_Time_Min'] = np.nan
orders.loc[(orders['Delivery_Distance_KM'] <= 0) | (orders['Delivery_Distance_KM'] > 50), 'Delivery_Distance_KM'] = np.nan
orders.loc[(orders['Customer_Rating'] < 1) | (orders['Customer_Rating'] > 5), 'Customer_Rating'] = np.nan

for col in ['Delivery_Time_Min', 'Delivery_Distance_KM', 'Customer_Rating']:
    orders[col] = orders[col].fillna(orders[col].median())

print(orders[['Delivery_Time_Min', 'Delivery_Distance_KM', 'Food_Amount', 'Customer_Rating']].describe())

In [ ]:
# AFTER fixing: validation charts
fig, ax = plt.subplots(2, 3, figsize=(16, 9))

ax[0,0].hist(orders['Delivery_Time_Min'], bins=20);       ax[0,0].set_title('Delivery Time (min)')
ax[0,1].hist(orders['Delivery_Distance_KM'], bins=20);    ax[0,1].set_title('Delivery Distance (KM)')
ax[0,2].hist(orders['Food_Amount'], bins=20);             ax[0,2].set_title('Food Amount')
ax[1,0].boxplot(orders['Delivery_Time_Min']);             ax[1,0].set_title('Delivery Time boxplot')
ax[1,1].scatter(orders['Delivery_Distance_KM'], orders['Delivery_Time_Min'], s=8)
ax[1,1].set_title('Distance vs Delivery Time')
orders['Customer_Rating'].round().value_counts().sort_index().plot(kind='bar', ax=ax[1,2])
ax[1,2].set_title('Customer Rating (rounded)')

plt.tight_layout()
plt.show()

## 6. Export processed CSVs (for Power BI)

In [ ]:
cust.to_csv('customers_clean.csv', index=False)
rest.to_csv('restaurants_clean.csv', index=False)
orders.to_csv('orders_clean.csv', index=False)

print('Saved:', cust.shape, rest.shape, orders.shape)
print(orders.isna().sum().sum(), 'missing values left in orders')

## Power BI DAX measures (Q2)
```
Total Orders = COUNTROWS(orders_clean)
Total Food Amount = SUM(orders_clean[Food_Amount])
Total Discount = SUM(orders_clean[Discount])
Average Delivery Time = AVERAGE(orders_clean[Delivery_Time_Min])
Average Customer Rating = AVERAGE(orders_clean[Customer_Rating])
```
**Relationships:** `customers_clean[Customer_ID]` -> `orders_clean[Customer_ID]` and `restaurants_clean[Restaurant_ID]` -> `orders_clean[Restaurant_ID]` (both one to many).